<a href="https://colab.research.google.com/github/vneelima44/Career-Intelligence-Bot/blob/main/notebooks/fairlens_bootstrap_cis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 17 — Bootstrap CIs across sensitivity scenarios
# ══════════════════════════════════════════════════════════════════
# Extends cell16 to 3 economic scenarios:
#   1. Base       (LGD=0.40, rate=3.0%)  — same as cell16, used for sanity check
#   2. High rate  (LGD=0.40, rate=5.0%)
#   3. High LGD   (LGD=0.55, rate=3.0%)
# Threshold fixed at 0.5 throughout.
# Output: artifacts/bootstrap_cis_sensitivity.json
# Runtime: ~15-25 min for 1000 iterations × 3 scenarios × 3 models.
# ══════════════════════════════════════════════════════════════════

import json
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# ── Load artifacts ─────────────────────────────────────────────────
test_probs   = pd.read_parquet('/content/test_probs.parquet')
test_meta    = pd.read_parquet('/content/test_meta.parquet')
cost_meta_df = pd.read_parquet('/content/cost_meta.parquet')

y_true_arr = test_meta['y_true'].values.astype(int)
race_arr   = test_meta['race_binary'].values.astype(int)
loans_arr  = cost_meta_df['loan_amount'].values.astype(float)
rates_arr  = cost_meta_df['interest_rate'].values.astype(float)
terms_arr  = cost_meta_df['loan_term_years'].values.astype(float)

n_samples = len(test_meta)
print(f"Loaded {n_samples:,} test records, {len(test_probs.columns)} models")


# ── Vectorized cost functions (identical to cell16) ────────────────
def amortized_interest_npv_vec(loans, rates, terms, discount_rate):
    r = rates / 100.0 / 12.0
    d = discount_rate / 12.0
    n = (terms * 12).astype(int)
    valid = (r > 0) & (n > 0)
    npv = np.zeros_like(loans, dtype=float)

    one_plus_r_n = np.zeros_like(loans, dtype=float)
    one_plus_r_n[valid] = (1 + r[valid]) ** n[valid]
    mp = np.zeros_like(loans, dtype=float)
    mp[valid] = (
        loans[valid] * (r[valid] * one_plus_r_n[valid])
        / (one_plus_r_n[valid] - 1)
    )
    npv[valid] = (
        mp[valid] * (1 - (1 + d) ** (-n[valid])) / d
        - loans[valid]
    )
    return npv


def remaining_balance_vec(loans, rates, terms, default_years):
    r = rates / 100.0 / 12.0
    n = (terms * 12).astype(int)
    k = (default_years * 12).astype(int)
    valid = (r > 0) & (n > 0)
    rb = loans.copy().astype(float)

    one_plus_r_n = np.zeros_like(loans, dtype=float)
    one_plus_r_n[valid] = (1 + r[valid]) ** n[valid]
    mp = np.zeros_like(loans, dtype=float)
    mp[valid] = (
        loans[valid] * (r[valid] * one_plus_r_n[valid])
        / (one_plus_r_n[valid] - 1)
    )

    one_plus_r_k = np.zeros_like(loans, dtype=float)
    one_plus_r_k[valid] = (1 + r[valid]) ** k[valid]
    rb[valid] = np.maximum(
        loans[valid] * one_plus_r_k[valid]
        - mp[valid] * (one_plus_r_k[valid] - 1) / r[valid],
        0.0,
    )
    return rb


def compute_metrics_indexed(idx, probs_array, threshold, lgd, discount_rate):
    preds = (probs_array[idx] >= threshold).astype(int)
    yt    = y_true_arr[idx]
    race  = race_arr[idx]
    l     = loans_arr[idx]
    r_    = rates_arr[idx]
    t_    = terms_arr[idx]

    results = {}
    for group, label in [(1, 'White'), (0, 'NonWhite')]:
        mask = race == group
        if not mask.any():
            return None

        ytg, ypg = yt[mask], preds[mask]
        lg, rg, tg = l[mask], r_[mask], t_[mask]
        n_g = int(mask.sum())

        fn_mask = (ytg == 1) & (ypg == 0)
        fp_mask = (ytg == 0) & (ypg == 1)

        fn_cost = 0.0
        if fn_mask.any():
            fn_cost = float(amortized_interest_npv_vec(
                lg[fn_mask], rg[fn_mask], tg[fn_mask], discount_rate
            ).sum())

        fp_cost = 0.0
        if fp_mask.any():
            fp_cost = float((remaining_balance_vec(
                lg[fp_mask], rg[fp_mask], tg[fp_mask], tg[fp_mask] / 2.0
            ) * lgd).sum())

        results[label] = {
            'approval_rate': float(ypg.mean()),
            'per_applicant': (fn_cost + fp_cost) / n_g if n_g > 0 else 0.0,
        }

    di_denom = max(results['White']['approval_rate'], 1e-9)
    return {
        'di':  results['NonWhite']['approval_rate'] / di_denom,
        'gap': results['NonWhite']['per_applicant'] - results['White']['per_applicant'],
    }


# ── Scenarios ──────────────────────────────────────────────────────
SCENARIOS = [
    ('Base',      0.40, 0.030),
    ('High rate', 0.40, 0.050),
    ('High LGD',  0.55, 0.030),
]
N_ITER    = 1000
THRESHOLD = 0.5

probs_arrays = {m: test_probs[m].values for m in test_probs.columns}

# Use the SAME bootstrap indices for all scenarios so CIs are comparable
# (variance from sampling only, not from different random draws).
np.random.seed(42)
all_indices = [np.random.choice(n_samples, n_samples, replace=True)
               for _ in range(N_ITER)]


# ── Run bootstrap for each scenario ────────────────────────────────
output = {'metadata': {'n_iterations': N_ITER, 'threshold': THRESHOLD,
                       'note': 'Same bootstrap indices used across scenarios'},
          'scenarios': {}}

for scen_name, lgd_s, disc_s in SCENARIOS:
    print(f"\n=== Scenario: {scen_name} (LGD={lgd_s}, rate={disc_s}) ===")
    results_by_model = {m: {'di': [], 'gap': []} for m in test_probs.columns}

    for idx in tqdm(all_indices):
        for m in test_probs.columns:
            out = compute_metrics_indexed(idx, probs_arrays[m], THRESHOLD, lgd_s, disc_s)
            if out is not None:
                results_by_model[m]['di'].append(out['di'])
                results_by_model[m]['gap'].append(out['gap'])

    # Point estimates
    full_idx = np.arange(n_samples)
    point_ests = {
        m: compute_metrics_indexed(full_idx, probs_arrays[m], THRESHOLD, lgd_s, disc_s)
        for m in test_probs.columns
    }

    scen_data = {'lgd': lgd_s, 'discount': disc_s, 'models': {}}
    for m in test_probs.columns:
        gap_vals = np.array(results_by_model[m]['gap'])
        di_vals  = np.array(results_by_model[m]['di'])
        scen_data['models'][m] = {
            'gap': {
                'point_estimate': float(point_ests[m]['gap']),
                'ci_lower':       float(np.percentile(gap_vals, 2.5)),
                'ci_upper':       float(np.percentile(gap_vals, 97.5)),
            },
            'di': {
                'point_estimate': float(point_ests[m]['di']),
                'ci_lower':       float(np.percentile(di_vals, 2.5)),
                'ci_upper':       float(np.percentile(di_vals, 97.5)),
            },
        }
    output['scenarios'][scen_name] = scen_data


# ── Save ───────────────────────────────────────────────────────────
with open('/content/bootstrap_cis.json', 'w') as f:
    json.dump(output, f, indent=2)

print(f"\n✅ Saved artifacts/bootstrap_cis_sensitivity.json")
print(f"\n{'='*70}")
print(f"SENSITIVITY SUMMARY — Gap per applicant by scenario")
print(f"{'='*70}")
for scen_name in [s[0] for s in SCENARIOS]:
    print(f"\n--- {scen_name} ---")
    for m in test_probs.columns:
        g = output['scenarios'][scen_name]['models'][m]['gap']
        print(f"  {m:25s} ${g['point_estimate']:+,.0f}  "
              f"[${g['ci_lower']:+,.0f}, ${g['ci_upper']:+,.0f}]")

Loaded 95,588 test records, 3 models

=== Scenario: Base (LGD=0.4, rate=0.03) ===


  0%|          | 0/1000 [00:00<?, ?it/s]


=== Scenario: High rate (LGD=0.4, rate=0.05) ===


  0%|          | 0/1000 [00:00<?, ?it/s]


=== Scenario: High LGD (LGD=0.55, rate=0.03) ===


  0%|          | 0/1000 [00:00<?, ?it/s]


✅ Saved artifacts/bootstrap_cis_sensitivity.json

SENSITIVITY SUMMARY — Gap per applicant by scenario

--- Base ---
  LR Baseline               $+4,711  [$+4,113, $+5,361]
  LR Manual RW              $+2,916  [$+2,400, $+3,464]
  NN + Focal Loss           $+2,333  [$+1,853, $+2,848]

--- High rate ---
  LR Baseline               $+4,587  [$+3,999, $+5,232]
  LR Manual RW              $-5,014  [$-5,764, $-4,250]
  NN + Focal Loss           $+555  [$-107, $+1,192]

--- High LGD ---
  LR Baseline               $+6,485  [$+5,679, $+7,379]
  LR Manual RW              $+3,532  [$+2,895, $+4,196]
  NN + Focal Loss           $+3,299  [$+2,665, $+3,989]
